In [ ]:
import pyspark
import dxpy
import re
import dxdata
from pyspark.sql.types import DateType
from pyspark.sql.functions import coalesce
sc = pyspark.SparkContext()

spark = pyspark.sql.SparkSession(sc)

In [ ]:
%load_ext autoreload
%autoreload 2


from recode_anno_filter import (extract_ldl, anno_presc_ldl)

In [ ]:
output_db_name = "arb_db"
db_uri = dxpy.find_one_data_object(name=f"{output_db_name}", classname="database", project=dxpy.PROJECT_CONTEXT_ID)['id']

In [ ]:
import hail as hl
hl.init(sc=sc, default_reference='GRCh38')

### Extract and clean LDL measurements:

Used Read2 and CTV3 codes:
- 44P6. - Serum LDL cholesterol level
- 44PI. - Calculated LDL cholesterol level
- 44PD. - Serum fasting LDL cholesterol level
- XaIp4 - Calculated LDL cholesterol level (only CTV3)

In [ ]:
final_ldl_ht = extract_ldl(spark)

In [ ]:
final_ldl_ht.count()

### Saving tb

In [ ]:
output_tb_name = "ldl_meas.ht"

In [ ]:
path = f"dnax://{db_uri}/{output_tb_name}"
final_ldl_ht.write(path, overwrite = True)

### Annotate with prescriptions

In [ ]:
ldl = hl.read_table(path)

In [ ]:
tb_name = "bp_prel.ht"
url = f"dnax://{db_uri}/{tb_name}"

bp = hl.read_table(url)

In [ ]:
ldl_meas = anno_presc_ldl(ldl, bp, db_uri)

In [ ]:
ldl_meas.export('ldl.tsv')